In [1]:
pip install flask pandas scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [23]:
from flask import Flask, render_template, request, redirect, session
import sqlite3
import joblib

app = Flask(__name__)
app.secret_key = "mysecretkey"
model = joblib.load("salary_model.pkl")

# Create database table
conn = sqlite3.connect("users.db")
conn.execute("""
CREATE TABLE IF NOT EXISTS users(
id INTEGER PRIMARY KEY AUTOINCREMENT,
username TEXT,
password TEXT
)
""")
conn.close()


@app.route("/", methods=["GET", "POST"])
def login():

    if request.method == "POST":

        username = request.form["username"]
        password = request.form["password"]

        conn = sqlite3.connect("users.db")
        cur = conn.cursor()

        cur.execute(
            "SELECT * FROM users WHERE username=? AND password=?",
            (username, password)
        )

        user = cur.fetchone()
        conn.close()

        if user:
            session["user"] = username
            return redirect("/home")

    return render_template("login.html")


@app.route("/register", methods=["GET", "POST"])
def register():

    if request.method == "POST":

        username = request.form["username"]
        password = request.form["password"]

        conn = sqlite3.connect("users.db")
        cur = conn.cursor()

        cur.execute(
            "INSERT INTO users(username,password) VALUES(?,?)",
            (username, password)
        )

        conn.commit()
        conn.close()

        return redirect("/")

    return render_template("register.html")


@app.route("/home", methods=["GET", "POST"])
def home():

    if "user" not in session:
        return redirect("/")

    prediction = None

    if request.method == "POST":

        experience = float(request.form["experience"])

        prediction = model.predict([[experience]])

        prediction = round(prediction[0], 2)

    return render_template(
        "home.html",
        prediction=prediction
    )


@app.route("/logout")
def logout():

    session.pop("user", None)

    return redirect("/")


if __name__ == "__main__":
    app.run(debug=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [05/Oct/2026 10:53:11] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2026 10:53:11] "GET /static/style.css HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2026 10:53:13] "POST / HTTP/1.1" 302 -
127.0.0.1 - - [05/Oct/2026 10:53:13] "GET /home HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2026 10:53:13] "GET /static/style.css HTTP/1.1" 304 -
C:\Users\James\anaconda3\Lib\site-packages\sklearn\base.py:464: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
127.0.0.1 - - [05/Oct/2026 10:53:17] "POST /home HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2026 10:53:17] "GET /static/style.css HTTP/1.1" 304 -
127.0.0.1 - - [05/Oct/2026 10:53:20] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [05/Oct/2026 10:53:20] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [05/Oct/2026 10:53:20] "GET /static/style.css HTTP/1.1" 304 -


In [6]:
import os

print("Current working directory:")
print(os.getcwd())

print("\nFiles in current folder:")
print(os.listdir())

print("\nFiles in templates folder:")
print(os.listdir("templates"))

Current working directory:
C:\Users\James\Untitled Folder

Files in current folder:
['.ipynb_checkpoints', 'app.ipynb', 'templates', 'users.db']

Files in templates folder:
['.ipynb_checkpoints', 'home.html', 'login.html', 'register.html', 'result.html']


In [8]:
register_html = """
<!DOCTYPE html>
<html>
<head>
    <title>Register</title>
</head>

<body>

<h2>Register Page</h2>

<form method="POST">

    <label>Username:</label>
    <input type="text" name="username" required>

    <br><br>

    <label>Password:</label>
    <input type="password" name="password" required>

    <br><br>

    <button type="submit">Register</button>

</form>

<br>

<a href="/">Back to Login</a>

</body>
</html>
"""

with open("templates/register.html", "w", encoding="utf-8") as f:
    f.write(register_html)

print("register.html created successfully")

register.html created successfully


In [9]:
import os

print(os.path.getsize("templates/register.html"), "bytes")

452 bytes


In [11]:
home_html = """
<!DOCTYPE html>
<html>
<head>
    <title>Home</title>
</head>

<body>

<h1>Welcome to the ML Application</h1>

<p>You have successfully logged in!</p>

<hr>

<h2>Salary Prediction</h2>

<p>ML prediction will be added here next.</p>

<br>

<a href="/logout">Logout</a>

</body>
</html>
"""

with open("templates/home.html", "w", encoding="utf-8") as f:
    f.write(home_html)

print("home.html created successfully")

home.html created successfully


In [13]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import joblib

# Sample training data
data = {
    "experience": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "salary": [25000, 30000, 35000, 40000, 50000,
               60000, 70000, 80000, 90000, 100000]
}

df = pd.DataFrame(data)

# Input and output
X = df[["experience"]]
y = df["salary"]

# Train model
model = LinearRegression()
model.fit(X, y)

# Save model
joblib.dump(model, "salary_model.pkl")

print("Model trained and saved successfully!")

Model trained and saved successfully!


In [14]:
import os

print(os.listdir())

['.ipynb_checkpoints', 'app.ipynb', 'salary_model.pkl', 'templates', 'users.db']


In [17]:
home_html = """
<!DOCTYPE html>
<html>
<head>
    <title>Salary Prediction</title>
</head>

<body>

<h1>Welcome to the ML Application</h1>

<p>You have successfully logged in!</p>

<hr>

<h2>Employee Salary Prediction</h2>

<form method="POST">

    <label>Years of Experience:</label>

    <input
        type="number"
        name="experience"
        step="0.1"
        min="0"
        required
    >

    <br><br>

    <button type="submit">
        Predict Salary
    </button>

</form>

{% if prediction is not none %}

<hr>

<h2>Prediction Result</h2>

<h3>Predicted Salary: ₹{{ prediction }}</h3>

{% endif %}

<br>

<a href="/logout">Logout</a>

</body>
</html>
"""

with open("templates/home.html", "w", encoding="utf-8") as f:
    f.write(home_html)

print("home.html updated successfully")

home.html updated successfully


In [19]:
import os

os.makedirs("static", exist_ok=True)

css = """
* {
    box-sizing: border-box;
    margin: 0;
    padding: 0;
}

body {
    font-family: Arial, Helvetica, sans-serif;
    background: #f4f7fb;
    color: #1f2937;
}

/* LOGIN / REGISTER */

.auth-page {
    min-height: 100vh;
    display: flex;
    justify-content: center;
    align-items: center;
    background: linear-gradient(135deg, #2563eb, #7c3aed);
}

.auth-card {
    width: 400px;
    background: white;
    padding: 40px;
    border-radius: 18px;
    box-shadow: 0 20px 50px rgba(0,0,0,0.15);
}

.logo {
    text-align: center;
    margin-bottom: 10px;
    font-size: 30px;
    font-weight: bold;
    color: #2563eb;
}

.subtitle {
    text-align: center;
    color: #6b7280;
    margin-bottom: 30px;
}

.form-group {
    margin-bottom: 20px;
}

.form-group label {
    display: block;
    margin-bottom: 8px;
    font-weight: 600;
}

.form-group input {
    width: 100%;
    padding: 13px;
    border: 1px solid #d1d5db;
    border-radius: 9px;
    font-size: 15px;
    outline: none;
}

.form-group input:focus {
    border-color: #2563eb;
    box-shadow: 0 0 0 3px rgba(37,99,235,0.1);
}

.btn {
    width: 100%;
    padding: 13px;
    border: none;
    border-radius: 9px;
    background: #2563eb;
    color: white;
    font-size: 16px;
    font-weight: bold;
    cursor: pointer;
}

.btn:hover {
    background: #1d4ed8;
}

.auth-link {
    text-align: center;
    margin-top: 22px;
    color: #6b7280;
}

.auth-link a {
    color: #2563eb;
    text-decoration: none;
    font-weight: bold;
}

/* DASHBOARD */

.navbar {
    height: 70px;
    background: white;
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 0 7%;
    box-shadow: 0 2px 10px rgba(0,0,0,0.06);
}

.nav-logo {
    font-size: 23px;
    font-weight: bold;
    color: #2563eb;
}

.logout {
    text-decoration: none;
    background: #fee2e2;
    color: #dc2626;
    padding: 9px 18px;
    border-radius: 8px;
    font-weight: bold;
}

.dashboard {
    max-width: 1100px;
    margin: 50px auto;
    padding: 0 25px;
}

.welcome {
    margin-bottom: 30px;
}

.welcome h1 {
    font-size: 32px;
    margin-bottom: 8px;
}

.welcome p {
    color: #6b7280;
}

.prediction-card {
    background: white;
    border-radius: 18px;
    padding: 35px;
    box-shadow: 0 10px 35px rgba(0,0,0,0.07);
    max-width: 700px;
}

.prediction-card h2 {
    margin-bottom: 10px;
}

.card-description {
    color: #6b7280;
    margin-bottom: 30px;
}

.predict-form {
    display: flex;
    gap: 15px;
    align-items: end;
}

.experience-field {
    flex: 1;
}

.experience-field label {
    display: block;
    margin-bottom: 8px;
    font-weight: 600;
}

.experience-field input {
    width: 100%;
    padding: 13px;
    border: 1px solid #d1d5db;
    border-radius: 9px;
    font-size: 16px;
}

.predict-btn {
    padding: 13px 25px;
    background: #2563eb;
    border: none;
    color: white;
    border-radius: 9px;
    font-size: 15px;
    font-weight: bold;
    cursor: pointer;
}

.predict-btn:hover {
    background: #1d4ed8;
}

.result-card {
    margin-top: 30px;
    padding: 25px;
    border-radius: 15px;
    background: #eff6ff;
    border: 1px solid #bfdbfe;
}

.result-label {
    color: #6b7280;
    margin-bottom: 8px;
}

.salary {
    font-size: 32px;
    font-weight: bold;
    color: #2563eb;
}

.icon {
    font-size: 35px;
    margin-bottom: 15px;
}

@media(max-width: 600px) {

    .auth-card {
        width: 90%;
        padding: 30px;
    }

    .navbar {
        padding: 0 20px;
    }

    .dashboard {
        margin-top: 30px;
    }

    .predict-form {
        flex-direction: column;
        align-items: stretch;
    }

    .predict-btn {
        width: 100%;
    }
}
"""

with open("static/style.css", "w", encoding="utf-8") as f:
    f.write(css)

print("style.css created successfully")

style.css created successfully


In [20]:
login_html = """
<!DOCTYPE html>
<html>

<head>
    <title>Login - SalaryAI</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>

<body>

<div class="auth-page">

    <div class="auth-card">

        <div class="logo">SalaryAI</div>

        <p class="subtitle">
            Employee Salary Prediction
        </p>

        <form method="POST">

            <div class="form-group">
                <label>Username</label>
                <input
                    type="text"
                    name="username"
                    placeholder="Enter your username"
                    required>
            </div>

            <div class="form-group">
                <label>Password</label>
                <input
                    type="password"
                    name="password"
                    placeholder="Enter your password"
                    required>
            </div>

            <button class="btn" type="submit">
                Login
            </button>

        </form>

        <div class="auth-link">
            Don't have an account?
            <a href="/register">Create account</a>
        </div>

    </div>

</div>

</body>
</html>
"""

with open("templates/login.html", "w", encoding="utf-8") as f:
    f.write(login_html)

print("login.html updated")

login.html updated


In [21]:
register_html = """
<!DOCTYPE html>
<html>

<head>
    <title>Register - SalaryAI</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>

<body>

<div class="auth-page">

    <div class="auth-card">

        <div class="logo">SalaryAI</div>

        <p class="subtitle">
            Create your account
        </p>

        <form method="POST">

            <div class="form-group">
                <label>Username</label>
                <input
                    type="text"
                    name="username"
                    placeholder="Choose a username"
                    required>
            </div>

            <div class="form-group">
                <label>Password</label>
                <input
                    type="password"
                    name="password"
                    placeholder="Create a password"
                    required>
            </div>

            <button class="btn" type="submit">
                Create Account
            </button>

        </form>

        <div class="auth-link">
            Already have an account?
            <a href="/">Login</a>
        </div>

    </div>

</div>

</body>
</html>
"""

with open("templates/register.html", "w", encoding="utf-8") as f:
    f.write(register_html)

print("register.html updated")

register.html updated


In [22]:
home_html = """
<!DOCTYPE html>
<html>

<head>
    <title>Dashboard - SalaryAI</title>
    <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
</head>

<body>

<nav class="navbar">

    <div class="nav-logo">
        SalaryAI
    </div>

    <a class="logout" href="/logout">
        Logout
    </a>

</nav>


<div class="dashboard">

    <div class="welcome">

        <h1>Salary Prediction Dashboard</h1>

        <p>
            Predict an employee's estimated salary using
            Machine Learning.
        </p>

    </div>


    <div class="prediction-card">

        <div class="icon">💼</div>

        <h2>Employee Salary Prediction</h2>

        <p class="card-description">
            Enter the employee's years of experience
            and our ML model will estimate the salary.
        </p>


        <form method="POST">

            <div class="predict-form">

                <div class="experience-field">

                    <label>
                        Years of Experience
                    </label>

                    <input
                        type="number"
                        name="experience"
                        step="0.1"
                        min="0"
                        placeholder="e.g. 5"
                        required>

                </div>

                <button
                    class="predict-btn"
                    type="submit">

                    Predict Salary

                </button>

            </div>

        </form>


        {% if prediction is not none %}

        <div class="result-card">

            <div class="result-label">
                Estimated Annual Salary
            </div>

            <div class="salary">
                ₹{{ "{:,.2f}".format(prediction) }}
            </div>

        </div>

        {% endif %}

    </div>

</div>

</body>

</html>
"""

with open("templates/home.html", "w", encoding="utf-8") as f:
    f.write(home_html)

print("home.html updated")

home.html updated
